In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

DATA_DIR = Path("data")

edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train = pd.read_csv(DATA_DIR / "train.csv")

## Data Prepocessing and Feature Engineering

The evaluation uses the 165 original transaction features, supplemented by in-degree, out-degree, total degree, PageRank, approximate betweenness centrality and 30 neighbour-average features. 

Missing values are filled using training-set medians, with zero used when a median is unavailable. Features are then standardised before SMOTE is applied to the training data to address class imbalance. For evaluation, preprocessing is fitted on the training portion only and validation data retain their original class distribution.

In [3]:
import networkx as nx

# Build directed graph: txId1 -> txId2
G = nx.from_pandas_edgelist(
    edges,
    source="txId1",
    target="txId2",
    create_using=nx.DiGraph(),
)

# Additional graph features
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
pagerank = nx.pagerank(G, alpha=0.85)
betweenness_samples = min(256, G.number_of_nodes())

print(
    f"Calculating approximate betweenness using "
    f"{betweenness_samples} sampled source nodes..."
)

betweenness = (
    nx.betweenness_centrality(
        G,
        k=betweenness_samples,
        normalized=True,
        endpoints=False,
        seed=42,
    )
    if G.number_of_nodes() > 2
    else dict.fromkeys(G.nodes, 0.0)
)

graph_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "pagerank",
    "betweenness",
]

# Apply the same graph features to train and test
for df in (train, test):
    df["in_degree"] = df["txId"].map(in_degrees).fillna(0)
    df["out_degree"] = df["txId"].map(out_degrees).fillna(0)
    df["total_degree"] = df["in_degree"] + df["out_degree"]
    df["pagerank"] = df["txId"].map(pagerank).fillna(0)
    df["betweenness"] = df["txId"].map(betweenness).fillna(0)

# Existing neighbour aggregation
# use both incoming and outgoing neighbours
G_undirected = G.to_undirected()

neighbor_source_cols = [f"feat_{i}" for i in range(1, 31)]
neighbor_cols = [f"neighbor_avg_feat_{i}" for i in range(1, 31)]

# use feature values from labelled training transactions only
train_labeled = train.loc[train["label"].isin([0, 1])]

node_features = dict(
    zip(
        train_labeled["txId"],
        train_labeled[neighbor_source_cols].to_numpy(),
    )
)


def aggregate_neighbor_features(node_ids):
    """Average 30 features across eligible immediate neighbours."""
    aggregated = np.zeros(
        (len(node_ids), len(neighbor_source_cols))
    )

    for row_index, node_id in enumerate(node_ids):
        if node_id not in G_undirected:
            continue

        neighbor_values = [
            node_features[neighbor]
            for neighbor in G_undirected.neighbors(node_id)
            if neighbor in node_features
        ]

        if neighbor_values:
            aggregated[row_index] = np.mean(
                neighbor_values, axis=0
            )

    return aggregated


# Use the same training-node lookup for both datasets
train[neighbor_cols] = aggregate_neighbor_features(train["txId"])
test[neighbor_cols] = aggregate_neighbor_features(test["txId"])

feature_cols = (
    [f"feat_{i}" for i in range(1, 166)]
    + graph_cols
    + neighbor_cols
)

print(f"Feature engineering complete: {len(feature_cols)} features.")

Calculating approximate betweenness using 256 sampled source nodes...
Feature engineering complete: 200 features.


In [4]:
from sklearn.model_selection import train_test_split

# Select labeled rows and features.
labeled_mask = train["label"].isin([0, 1])

X = train.loc[labeled_mask, feature_cols].copy()
y = train.loc[labeled_mask, "label"].astype(int)
X_test = test[feature_cols].copy()

# Split before learning imputation values.
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Learn medians only from the training subset.
medians = X_train.median().fillna(0)

X_train = X_train.fillna(medians)
X_val = X_val.fillna(medians)
X_test = X_test.fillna(medians)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)

Training: (24988, 200) (24988,)
Validation: (6247, 200) (6247,)


In [5]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

# Scale before SMOTE because neighbor distances depend on feature scales.
# Fit preprocessing on the real training rows only; transform test identically.
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=feature_cols, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=feature_cols, index=X_val.index,
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=feature_cols, index=X_test.index
)

# Generate synthetic examples for training only. Never resample test data.
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train
)

print("Before SMOTE:", y_train.value_counts().sort_index().to_dict())
print("After SMOTE:", y_train_resampled.value_counts().sort_index().to_dict())

Before SMOTE: {0: 22073, 1: 2915}
After SMOTE: {0: 22073, 1: 22073}


## Evaluation Metrics

In [6]:
import lightgbm as lgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score
)

In [8]:
models = {
    "Logistic Regression": LogisticRegression(class_weight=None, random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight=None, random_state=42, max_depth=10),
    "LightGBM (Ensemble)": lgb.LGBMClassifier(
        n_estimators=200, 
        learning_rate=0.05,  
        reg_alpha=0.1, # L1 regularization
        max_depth=7,
        class_weight=None,  
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight=None,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting (Ensemble)": HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,  
        max_depth=7, 
        class_weight=None,     # Training data is already balanced with SMOTE.
        early_stopping=False, 
        random_state=42,
)
}

best_model = None
best_auc = 0
best_model_name = ""
best_thresholds = {}

print("\nModel Evaluation on Validation Set:")

for name, model in models.items():
    model.fit(X_train_resampled, y_train_resampled)

    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)
    
    # Choose the threshold using validation labels only.
    thresholds = np.arange(0.10, 0.90, 0.05)
    f1_scores = [
        f1_score(
            y_val,
            (y_pred_proba >= threshold).astype(int),
            zero_division=0,
        )
        for threshold in thresholds
    ]

    best_thresh = float(thresholds[np.argmax(f1_scores)])
    best_thresholds[name] = best_thresh

    y_pred = (y_pred_proba >= best_thresh).astype(int)

    if auc > best_auc:
        best_auc = auc
        best_model = model
        best_model_name = name

    print(
        f"\n[{name}]"
        f"\n  Best Threshold: {best_thresh:.2f}"
        f"\n  Precision: {precision_score(y_val, y_pred, zero_division=0):.4f}"
        f"\n  Recall:    {recall_score(y_val, y_pred, zero_division=0):.4f}"
        f"\n  F1:        {f1_score(y_val, y_pred, zero_division=0):.4f}"
        f"\n  ROC-AUC:   {roc_auc_score(y_val, y_pred_proba):.4f}"
        f"\n  Accuracy:  {accuracy_score(y_val, y_pred):.4f}"
    )

print(f"\nFinal selected model: {best_model_name} (AUC: {best_auc:.4f})")


Model Evaluation on Validation Set:

[Logistic Regression]
  Best Threshold: 0.85
  Precision: 0.8194
  Recall:    0.8217
  F1:        0.8205
  ROC-AUC:   0.9785
  Accuracy:  0.9581

[Decision Tree]
  Best Threshold: 0.80
  Precision: 0.9003
  Recall:    0.9410
  F1:        0.9202
  ROC-AUC:   0.9803
  Accuracy:  0.9810
[LightGBM] [Info] Number of positive: 22073, number of negative: 22073
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012403 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 50076
[LightGBM] [Info] Number of data points in the train set: 44146, number of used features: 199
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

## Cross-validation 

It compares all five models in the existing `models` dictionary: Logistic Regression, Decision Tree, Random Forest, LightGBM and HistGradientBoosting. Each uses its currently defined configuration and the same five stratified folds of the original labelled `X` and `y` (not the resampled data). 

Within each fold, median imputation, scaling and SMOTE are fitted only on the training data to avoid leaking information from the validation fold. This comparison uses all labelled rows, including the earlier validation subset, so it should not be treated as an independent final test.

I also keep the existing graph features which were created before splitting the data. These features may include information from validation nodes, but not their target labels. The results therefore reflect a situation where the graph and node features are available together, rather than how well the models would perform on future or completely unseen graphs.


In [9]:
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {name: clone(model) for name, model in models.items()}

cv_results = []
for name, model in cv_models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("model", model),
    ])
    scores = cross_val_score(
        pipeline, X, y, cv=cv, scoring="roc_auc", n_jobs=1,
        error_score="raise",
    )
    cv_results.append({
        "Model": name,
        **{f"Fold {i + 1}": score for i, score in enumerate(scores)},
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std(ddof=1),
    })
    print(f"{name}: ROC-AUC = {scores.mean():.5f} ± {scores.std(ddof=1):.5f}")

cv_results = pd.DataFrame(cv_results).sort_values(
    "Mean ROC-AUC", ascending=False
)
cv_results.round(5)


Logistic Regression: ROC-AUC = 0.98008 ± 0.00174
Decision Tree: ROC-AUC = 0.97240 ± 0.00596
[LightGBM] [Info] Number of positive: 22072, number of negative: 22072
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010703 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 50303
[LightGBM] [Info] Number of data points in the train set: 44144, number of used features: 199
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warni

,Model,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean ROC-AUC,Std ROC-AUC
2,LightGBM (Ensemble),0.99880,0.99882,0.99787,0.99876,0.99880,0.99861,0.00041
4,HistGradientBoosting (Ensemble),0.99876,0.99889,0.99787,0.99857,0.99881,0.99858,0.00041
3,Random Forest,0.99816,0.99835,0.99803,0.99868,0.99827,0.99830,0.00024
0,Logistic Regression,0.97821,0.98260,0.98036,0.97867,0.98057,0.98008,0.00174
1,Decision Tree,0.96527,0.96805,0.97189,0.97850,0.97830,0.97240,0.00596


From the results, LightGBM (Ensemble) and HistGradientBoosting (Ensemble) are the highest 2 ranking models. Together with results are Kaggle, LightGBM (Ensemble) and HistGradientBoosting (Ensemble) are chosen as the final models.